# Workshop 7 — Denoising Autoencoder for Devnagari Digits
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Building a Convolutional Autoencoder to remove noise from Devnagari handwritten digit images.
Architecture: Encoder (compress) → Bottleneck → Decoder (reconstruct)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow: {tf.__version__}')
np.random.seed(42)
tf.random.set_seed(42)

## 1. What is an Autoencoder?

An autoencoder is a neural network that learns to:
1. **Encode** input into a compressed latent representation
2. **Decode** the latent representation back to the original

For **denoising**, we train it to reconstruct clean images from noisy inputs.

## 2. Load & Prepare Data

In [ ]:
# Use MNIST as proxy for Devnagari digits (same structure: 28x28 grayscale)
# Replace with Devnagari dataset when available
(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()

# Normalise to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# Add channel dimension: (N, 28, 28) -> (N, 28, 28, 1)
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Add Gaussian Noise

In [ ]:
def add_noise(images, noise_factor=0.4):
    """Add Gaussian noise to images."""
    noisy = images + noise_factor * np.random.normal(size=images.shape)
    return np.clip(noisy, 0.0, 1.0)

NOISE_FACTOR = 0.4
X_train_noisy = add_noise(X_train, NOISE_FACTOR)
X_test_noisy  = add_noise(X_test,  NOISE_FACTOR)

# Visualise clean vs noisy
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(X_test[i, :, :, 0], cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Clean', fontsize=8)
    axes[1, i].imshow(X_test_noisy[i, :, :, 0], cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Noisy', fontsize=8)
plt.suptitle(f'Clean vs Noisy Images (noise_factor={NOISE_FACTOR})', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Convolutional Autoencoder Architecture

In [ ]:
def build_denoising_autoencoder(input_shape=(28, 28, 1)):
    """
    Convolutional Autoencoder for image denoising.
    Encoder: Conv2D -> MaxPool (compress spatial dimensions)
    Decoder: Conv2D -> UpSampling (reconstruct original size)
    """
    inputs = keras.Input(shape=input_shape)

    # ── ENCODER ──────────────────────────────────────────────────────────
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)          # 28x28 -> 14x14

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.MaxPooling2D(2, padding='same')(x)    # 14x14 -> 7x7

    # ── BOTTLENECK ────────────────────────────────────────────────────────
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(encoded)

    # ── DECODER ──────────────────────────────────────────────────────────
    x = layers.UpSampling2D(2)(x)                          # 7x7 -> 14x14
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.UpSampling2D(2)(x)                          # 14x14 -> 28x28
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Output: reconstruct clean image
    decoded = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)

    autoencoder = keras.Model(inputs, decoded, name='Denoising_Autoencoder')
    return autoencoder

autoencoder = build_denoising_autoencoder()
autoencoder.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['mae']
)
autoencoder.summary()

## 5. Train the Autoencoder

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6)
]

history = autoencoder.fit(
    X_train_noisy, X_train,          # input: noisy, target: clean
    epochs=30,
    batch_size=128,
    validation_data=(X_test_noisy, X_test),
    callbacks=callbacks,
    verbose=1
)

print(f'Final val_loss: {min(history.history["val_loss"]):.4f}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train Loss', color='#EF4444', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#F59E0B', linewidth=2, linestyle='--')
axes[0].set_title('Autoencoder Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', color='#0EA5E9', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#6366F1', linewidth=2, linestyle='--')
axes[1].set_title('Mean Absolute Error'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Denoising Autoencoder Training', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Denoising Results

In [ ]:
# Reconstruct denoised images
denoised = autoencoder.predict(X_test_noisy[:10])

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
for i in range(10):
    axes[0, i].imshow(X_test[i, :, :, 0], cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Clean', fontsize=10)

    axes[1, i].imshow(X_test_noisy[i, :, :, 0], cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('Noisy', fontsize=10)

    axes[2, i].imshow(denoised[i, :, :, 0], cmap='gray')
    axes[2, i].axis('off')
    if i == 0: axes[2, i].set_ylabel('Denoised', fontsize=10)

plt.suptitle('Clean → Noisy → Denoised', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# PSNR metric
mse_noisy = np.mean((X_test[:10] - X_test_noisy[:10])**2)
mse_denoised = np.mean((X_test[:10] - denoised)**2)
psnr_noisy = 10 * np.log10(1.0 / mse_noisy)
psnr_denoised = 10 * np.log10(1.0 / mse_denoised)
print(f'PSNR (noisy):    {psnr_noisy:.2f} dB')
print(f'PSNR (denoised): {psnr_denoised:.2f} dB  (+{psnr_denoised-psnr_noisy:.2f} dB improvement)')